In [6]:
import pandas as pd

df = pd.read_csv('../data/datas.tsv',sep='\t',header=None,names=['text','label','id'])
print(df.shape)
df.head()

(43410, 3)


,text,label,id
0,My favourite food is anything I didn't have to...,27,eebbqej
1,"Now if he does off himself, everyone will thin...",27,ed00q6i
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj
3,To make her feel threatened,14,ed7ypvh
4,Dirty Southern Wankers,3,ed0bdzj


In [7]:
multi_label = df['label'].astype(str).str.contains(',')
print("Multi Label rows:",multi_label.sum(),"out of", len(df))

df =df[~multi_label].copy()
df['label']=df['label'].astype(int)
print("Remaining single-label rows:", df.shape[0])

Multi Label rows: 7102 out of 43410
Remaining single-label rows: 36308


In [9]:
for label in sorted(df['label'].unique()):
    print(f"label {label}:")
    print(df[df['label']==label]['text'].iloc[0])
    print()

label 0:
Damn youtube and outrage drama is super lucrative for reddit

label 0,1:
LOL. Super cute!

label 0,1,11:
Good read man, sounds like a crazy time, its the worst vomiting red in front of ramdom people lmfao 😂

label 0,1,12:
Sounds awesome, but I am so out of practice I'd be embarrassed to show up lol.

label 0,1,13:
Sounds like a horror junkie in the making. Congrats dad, you’ll have a buddy to watch movies with in a year or two!

label 0,1,17:
Klokslag 12 I find really enjoyable, horror movie podcast, Im not a horror fan but I find these dudes very entertaining

label 0,1,18:
Wow... That looks great! Would love to get some actual vintage jerseys like that someday. Keep my eyes on eBay. Lol

label 0,1,4:
Yes but [NAME] advice is better then some randoms lol he was one of the leagues best ever defenders

label 0,1,7:
Pretty good, yourself? Edited - Saw the link, hahaha... How you doin? ;)

label 0,10:
Because the content creators don't deserve to be paid, your seconds spent liste

In [8]:
label_to_emotion = {
    0: 'admiration', 1: 'amusement', 2: 'anger', 3: 'annoyance', 4: 'approval',
    5: 'caring', 6: 'confusion', 7: 'curiosity', 8: 'desire', 9: 'disappointment',
    10: 'disapproval', 11: 'disgust', 12: 'embarrassment', 13: 'excitement', 14: 'fear',
    15: 'gratitude', 16: 'grief', 17: 'joy', 18: 'love', 19: 'nervousness',
    20: 'optimism', 21: 'pride', 22: 'realization', 23: 'relief', 24: 'remorse',
    25: 'sadness', 26: 'surprise', 27: 'neutral'
}

df['emotion'] = df['label'].map(label_to_emotion)
df['emotion'].value_counts()

emotion
neutral           12823
admiration         2710
approval           1873
gratitude          1857
amusement          1652
annoyance          1451
love               1427
disapproval        1402
curiosity          1389
anger              1025
optimism            861
confusion           858
joy                 853
sadness             817
surprise            720
disappointment      709
caring              649
realization         586
excitement          510
disgust             498
fear                430
desire              389
remorse             353
embarrassment       203
relief               88
nervousness          85
pride                51
grief                39
Name: count, dtype: int64

In [9]:
import sys
!{sys.executable} -m pip install nltk -q --break-system-packages

In [10]:
import nltk
nltk.download("stopwords")

/run/media/biplob/e1a9a613-7e49-4b9e-81ed-a1c7ad2b4cae/biplob/Emotion_analysis/jupyter-venv/lib/python3.14/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/biplob/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package stopwords to /home/biplob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
for negation in ('not', 'no', 'nor', 'never'):
    stop_words.discard(negation)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)
    

In [17]:
df['clean_text'] =df['text'].apply(clean_text)
df =df[df['clean_text'].str.len() > 0]
df.head()

,text,label,id,emotion,clean_text
0,My favourite food is anything I didn't have to...,27,eebbqej,neutral,favourite food anything didnt cook
1,"Now if he does off himself, everyone will thin...",27,ed00q6i,neutral,everyone think hes laugh screwing people inste...
2,WHY THE FUCK IS BAYLESS ISOING,2,eezlygj,anger,fuck bayless isoing
3,To make her feel threatened,14,ed7ypvh,fear,make feel threatened
4,Dirty Southern Wankers,3,ed0bdzj,annoyance,dirty southern wankers


In [18]:
sample = df['text'].iloc[0]
print("Before:",sample)
print("After:",clean_text(sample))

Before: My favourite food is anything I didn't have to cook myself.
After: favourite food anything didnt cook


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorize =  TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X= vectorize.fit_transform(df['clean_text'])

print(X.shape)

(36235, 20000)


In [20]:
from sklearn.model_selection import train_test_split

Y = df['label']
X_train,X_text,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

print("Train size:", X_train.shape)
print("Test size:",X_text.shape)

Train size: (28988, 20000)
Test size: (7247, 20000)


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

lr_model = LogisticRegression(max_iter=2000, class_weight='balanced')
lr_model.fit(X_train, Y_train)

lr_pred = lr_model.predict(X_text)
lr_accuracy = lr_model.score(X_text, Y_test)
lr_macro_f1 = f1_score(Y_test, lr_pred, average='macro')

print("Logistic Regression accuracy:", round(lr_accuracy, 4))
print("Logistic Regression macro F1:", round(lr_macro_f1, 4))
print()
target_names = [label_to_emotion[i] for i in sorted(label_to_emotion)]
print(classification_report(Y_test, lr_pred, target_names=target_names, zero_division=0))

Logistic Regression accuracy: 0.4111
Logistic Regression macro F1: 0.3857

                precision    recall  f1-score   support

    admiration       0.62      0.62      0.62       542
     amusement       0.74      0.77      0.76       330
         anger       0.40      0.51      0.45       205
     annoyance       0.16      0.15      0.15       290
      approval       0.29      0.32      0.31       374
        caring       0.22      0.45      0.29       130
     confusion       0.28      0.43      0.34       171
     curiosity       0.15      0.22      0.18       275
        desire       0.30      0.67      0.41        78
disappointment       0.12      0.26      0.16       142
   disapproval       0.20      0.41      0.27       280
       disgust       0.34      0.61      0.43       100
 embarrassment       0.35      0.59      0.44        41
    excitement       0.21      0.40      0.28       102
          fear       0.44      0.69      0.54        86
     gratitude       0.94   

In [22]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_model.fit(X_train, Y_train)

dt_pred = dt_model.predict(X_text)
dt_accuracy = dt_model.score(X_text, Y_test)
dt_macro_f1 = f1_score(Y_test, dt_pred, average='macro')

print("Decision Tree accuracy:", round(dt_accuracy, 4))
print("Decision Tree macro F1:", round(dt_macro_f1, 4))

Decision Tree accuracy: 0.408
Decision Tree macro F1: 0.3437


In [23]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(class_weight='balanced', max_iter=5000)
svm_model.fit(X_train, Y_train)

svm_pred = svm_model.predict(X_text)
svm_accuracy = svm_model.score(X_text, Y_test)
svm_macro_f1 = f1_score(Y_test, svm_pred, average='macro')

print("Linear SVM accuracy:", round(svm_accuracy, 4))
print("Linear SVM macro F1:", round(svm_macro_f1, 4))

Linear SVM accuracy: 0.4301
Linear SVM macro F1: 0.3594


In [ ]:
import pickle

# Update this to whichever model actually performed best above
best_model = lr_model

with open('../models/my_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
with open('../models/my_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorize, f)
with open('../models/label_mapping.pkl', 'wb') as f:
    pickle.dump(label_to_emotion, f)

In [4]:
import pickle

with open('../models/my_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('../models/my_vectorizer.pkl', 'rb') as f:
    loaded_vectorizer = pickle.load(f)

with open('../models/label_mapping.pkl', 'rb') as f:
    loaded_mapping = pickle.load(f)

def predict_emotion(text):
    cleaned = clean_text(text)
    vector = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vector)[0]
    return loaded_mapping[prediction]

print(predict_emotion("I am so happy today, everything is amazing"))
print(predict_emotion("I feel scared and anxious about tomorrow"))

joy
fear


In [5]:
import pickle

with open('../models/my_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('../models/my_vectorizer.pkl', 'rb') as f:
    loaded_vectorize = pickle.load(f)
with open('../models/label_mapping.pkl', 'rb') as f:
    loaded_mapping = pickle.load(f)

def predict_emotion(text):
    cleaned = clean_text(text)
    vector = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vector)[0]
    return loaded_mapping[prediction]

print(predict_emotion("I am so happy today, everything is amazing"))
print(predict_emotion("I feel scared and anxious about tomorrow"))
print(predict_emotion("Thank you so much, I really appreciate it"))
print(predict_emotion("I can't believe this happened, I'm furious"))

joy
fear
joy
joy
